# Qwen2.5-1.5B-Instruct + QLoRA: a generative model on the same task as PhoBERT v7

Setup: **GPU T4** (one is enough), **Internet On**, secret `HF_TOKEN`.
**Input:** the same private dataset as the v7 notebook (`v7_train.csv`, `v7_val.csv`, columns `text,label`).

The base model is loaded in 4-bit and only LoRA adapters are trained (about 1% of the weights). The prompt holds
the annotation guideline and the comment; the model learns to answer with the label word. Two epochs, the best
epoch on validation is kept and the adapter is pushed to `<your HF name>/qwen-vi-toxic-lora`.
The held-out test half is not on Kaggle: it is scored afterwards with `scripts/compare_generative.py`.
Takes roughly 30-60 minutes.

In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -U peft bitsandbytes accelerate
!pip install -q --no-deps -e .
import glob, os, shutil, json
os.makedirs("data/adapt_v6", exist_ok=True)
for split, name in [("train", "v7_train.csv"), ("val", "v7_val.csv")]:
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert len(matches) == 1, f"Expected exactly one {name}, found: {matches}"
    shutil.copy(matches[0], f"data/adapt_v6/{split}.csv")

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from huggingface_hub import whoami
hf_user = whoami()["name"]
print("Logged in as:", hf_user)

In [ ]:
!python scripts/train_qwen_qlora.py --data-dir data/adapt_v6 --output-dir models/qwen_qlora --epochs 2 --push {hf_user}/qwen-vi-toxic-lora

Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps(json.load(open("reports/qwen_qlora_results.json")), ensure_ascii=False))